# Bridge 02 — einsum, Linear Algebra, Scatter Ops

Mechanics that block dojo 05 (einsum patterns) and dojo 06 (SVD, scatter-reduce, TP/FP masks).  
No hints. 3–8 lines per problem.

In [71]:
import numpy as np
np.random.seed(0)

---
## Section A — einsum from problem descriptions

For each problem, write **exactly one `np.einsum` call**. No other ops allowed.

In [72]:
rand_arr = np.ones(shape=(3,4))
rand_arr

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [84]:
## Transpose
assert np.allclose(rand_arr.T, np.einsum('ij->ji', rand_arr)), rand_arr.T

print("Summation all elements added:", np.einsum('ij->', rand_arr), np.einsum('ij->', rand_arr).shape)

print(f"Column Sum:{np.einsum("ij->j", rand_arr)} and Row Sum {np.einsum("ij->i", rand_arr)}")

print(f"Dot Product with Matrix: \n {np.einsum('ij,ij->ij', rand_arr, rand_arr)}")

Summation all elements added: 12.0 ()
Column Sum:[3. 3. 3. 3.] and Row Sum [4. 4. 4.]
Dot Product with Matrix: 
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]


In [74]:
rand_arr_1 = np.random.uniform(size=(3,4))
rand_arr_2= np.random.normal(size=(7,4))
print(rand_arr_1.shape, rand_arr_2.shape)
print("Matrix Multiplication:") 
print("Einsum:", np.einsum("ij, kj-> ik", rand_arr_1, rand_arr_2).shape)
print("Operator:", (rand_arr_1@rand_arr_2.T).shape)
assert np.allclose(np.einsum("ij, kj-> ik", rand_arr_1, rand_arr_2), rand_arr_1@rand_arr_2.T), "Same operation"

print("Outer Product:", np.einsum("ij,kl-> ijkl", rand_arr_1, rand_arr_2).shape)

(3, 4) (7, 4)
Matrix Multiplication:
Einsum: (3, 7)
Operator: (3, 7)
Outer Product: (3, 4, 7, 4)


In [75]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print(np.einsum('ii->i', A))   # [1 5 9]


[1 5 9]


### A1 — matrix-vector product
`A` is `(M, N)`, `v` is `(N,)`. Compute `A @ v` → `(M,)`.

In [76]:
A = np.random.randn(4, 5)
v = np.random.randn(5)

result = np.einsum('ik,k->i', A, v)
print(result)

assert result.shape == (4,)
assert np.allclose(result, A @ v), result

[-0.85488565  2.86111887  2.91151766  0.47275281]


### A2 — row-wise dot products
`A` and `B` are both `(N, D)`. Compute the dot product of each row pair → shape `(N,)`.

In [83]:
A = np.random.randn(6, 4)
B = np.random.randn(6, 4)

result = np.einsum('ij,ij->i', A, B)

assert result.shape == (6,)
assert np.allclose(result, (A * B).sum(axis=1)), result

### A3 — outer product
`u` is `(M,)`, `v` is `(N,)`. Compute the outer product → shape `(M, N)`.

In [85]:
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6., 7.])

result = np.einsum('i,j ->ij', u, v)  # YOUR CODE HERE

assert result.shape == (3, 4)
assert np.allclose(result, np.outer(u, v)), result

### A4 — equivalent of A @ B.T
`A` is `(N, D)`, `B` is `(M, D)`. Compute `A @ B.T` → shape `(N, M)` using einsum.

In [86]:
A = np.random.randn(3, 5)
B = np.random.randn(4, 5)

result = np.einsum('ij,kj ->ik', A, B)  # YOUR CODE HERE

assert result.shape == (3, 4)
assert np.allclose(result, A @ B.T), result

### A5 — batch matmul
`A` is `(B, N, M)`, `C` is `(B, M, K)`. Multiply each pair in the batch → `(B, N, K)`.

In [92]:
A = np.random.randn(5, 3, 4)
C = np.random.randn(5, 4, 6)

result = np.einsum('ijk,ikl -> ijl', A, C)  # YOUR CODE HERE

assert result.shape == (5, 3, 6)
ref = np.stack([A[i] @ C[i] for i in range(5)])
assert np.allclose(result, ref), result

### A6 — batch trace
`S` is `(B, N, N)`. Compute the trace of each matrix in the batch → `(B,)`.

In [101]:
S = np.random.randn(4, 5, 5)

result = np.einsum('bii -> b', S)  # YOUR CODE HERE

assert result.shape == (4,)
ref = np.array([np.trace(S[i]) for i in range(4)])
assert np.allclose(result, ref), result

### A7 — element-wise product then sum last axis
`A` and `B` are `(B, N, D)`. Element-wise multiply, then sum over last axis → `(B, N)`.

In [102]:
A = np.random.randn(3, 4, 5)
B = np.random.randn(3, 4, 5)

result = np.einsum('bij,bij -> bi', A, B)  # YOUR CODE HERE

assert result.shape == (3, 4)
assert np.allclose(result, (A * B).sum(axis=-1)), result

### A8 — scaled dot-product: `Q @ K.T / sqrt(d_k)`
`Q` is `(T, d_k)`, `K` is `(T, d_k)`. Compute attention scores → `(T, T)`.

In [112]:
T, d_k = 6, 8
Q = np.random.randn(T, d_k)
K = np.random.randn(T, d_k)

scores = np.einsum('id,jd -> ij', Q, K) /np.sqrt(d_k) # YOUR CODE HERE

assert scores.shape == (T, T)
assert np.allclose(scores, (Q @ K.T) / np.sqrt(d_k)), scores

---
## Section B — SVD & Linear Algebra

### B1 — SVD decomposition
Decompose `A` of shape `(6, 4)` using `np.linalg.svd` with `full_matrices=False`. Verify `U @ np.diag(s) @ Vt` reconstructs `A`.

In [114]:
A = np.random.randn(6, 4)

U, s, Vt = np.linalg.svd(A, full_matrices=False) # YOUR CODE HERE

assert U.shape  == (6, 4), U.shape
assert s.shape  == (4,),   s.shape
assert Vt.shape == (4, 4), Vt.shape
assert np.allclose(U @ np.diag(s) @ Vt, A, atol=1e-10), 'Reconstruction failed'

### B2 — low-rank reconstruction
Given SVD of `A` (from B1), reconstruct using only the top `k=2` singular values/vectors.

In [ ]:
k = 2

A_k = None  # YOUR CODE HERE

assert A_k.shape == (6, 4)
# Rank-k should be worse than full rank
err_k    = np.linalg.norm(A - A_k, 'fro')
err_full = np.linalg.norm(A - U @ np.diag(s) @ Vt, 'fro')
assert err_k > err_full, 'rank-k approximation should have higher error than full rank'
assert err_k < np.linalg.norm(A, 'fro'), 'rank-k should be better than zero'

### B3 — SVD explained variance
Singular values relate to variance: `var_i = s_i^2 / sum(s^2)`. Compute explained variance ratio and cumulative sum. Verify sum = 1.

In [ ]:
evr  = None  # YOUR CODE HERE
cumr = None  # YOUR CODE HERE

assert evr.shape  == (4,)
assert cumr.shape == (4,)
assert np.isclose(evr.sum(), 1.0, atol=1e-6)
assert np.isclose(cumr[-1],  1.0, atol=1e-6)
assert np.all(evr >= 0)
assert evr[0] >= evr[1], 'should be sorted descending'

### B4 — verify orthogonality of U
U from SVD should satisfy `U.T @ U = I`. Check this explicitly.

In [ ]:
gram = None  # YOUR CODE HERE

assert gram.shape == (4, 4)
assert np.allclose(gram, np.eye(4), atol=1e-10), f'U.T @ U not identity:\n{gram}'

### B5 — solve a linear system
Solve `Ax = b` using `np.linalg.solve`. Then verify `A @ x ≈ b`.

In [ ]:
A_sq = np.array([[2., 1.], [5., 3.]])
b    = np.array([8., 19.])

x = None  # YOUR CODE HERE

assert x.shape == (2,)
assert np.allclose(A_sq @ x, b, atol=1e-8), f'A@x = {A_sq @ x}, b = {b}'

---
## Section C — Boolean Logic Masks (TP/FP/FN pattern)

### C1
Given `y_true` and `y_pred` of shape `(N,)` with values in `{0,1}`, compute:
- `TP`: true predicted as true
- `FP`: false predicted as true
- `FN`: true predicted as false
- `TN`: false predicted as false

All as scalars. No loops.

In [ ]:
y_true = np.array([1,0,1,1,0,0,1,0,1,0])
y_pred = np.array([1,0,0,1,1,0,1,0,0,0])

TP = None  # YOUR CODE HERE
FP = None  # YOUR CODE HERE
FN = None  # YOUR CODE HERE
TN = None  # YOUR CODE HERE

assert TP == 3, TP
assert FP == 1, FP
assert FN == 2, FN
assert TN == 4, TN
assert TP + FP + FN + TN == len(y_true)

### C2 — per-class TP for multiclass
Given `y_true` and `y_pred` of shape `(N,)` with values in `{0,...,C-1}`, compute `TP[c]` for all classes at once — shape `(C,)`. No loop over classes.

In [ ]:
C = 4
y_true = np.array([0, 1, 2, 0, 1, 3, 2, 0, 1, 2])
y_pred = np.array([0, 1, 1, 0, 2, 3, 2, 1, 1, 2])

# Hint: for class c, TP[c] = sum((y_true == c) & (y_pred == c))
# Can you do all classes at once without a for loop?
TP_per_class = None  # YOUR CODE HERE

assert TP_per_class.shape == (4,)
assert np.allclose(TP_per_class, [2, 2, 2, 1]), TP_per_class

### C3 — compound boolean conditions
Given `scores` and `labels` arrays of shape `(N,)`, find the **indices** of samples that are:
- Positive class (`label == 1`)
- AND high confidence (`score > 0.7`)
- AND correctly classified (i.e., score > 0.5 for positive class)

In [ ]:
scores = np.array([0.9, 0.3, 0.8, 0.6, 0.2, 0.75, 0.55, 0.85])
labels = np.array([1,   0,   1,   0,   1,   1,    1,    1   ])

# positive, high-conf, and score > 0.5
good_idx = None  # YOUR CODE HERE

assert np.allclose(good_idx, [0, 2, 5, 7]), good_idx

---
## Section D — Log-Sum-Exp & Numerical Stability

### D1 — why naive exp overflows
Compute `np.exp([1000., 1001., 1002.])`. Observe the output. Then compute `np.exp([1000., 1001., 1002.] - 1002.)` and compare the ratio of the second to the third element in both cases.

In [ ]:
x = np.array([1000., 1001., 1002.])

naive  = None  # YOUR CODE HERE
stable = None  # YOUR CODE HERE

print('naive:', naive)    # expect inf
print('stable:', stable)  # expect finite

# Ratios should be identical
# naive[1]/naive[2] is nan/inf — undefined
# stable[1]/stable[2] should be exp(-1)
assert np.isclose(stable[1] / stable[2], np.exp(-1), atol=1e-8)

### D2 — stable log-sum-exp for a vector
Implement `log(sum(exp(x)))` for `x` of any shape along axis=-1. Use the max subtraction trick.

In [ ]:
def logsumexp(x):
    """x: any shape, reduce over last axis. Returns shape = x.shape[:-1]"""
    pass  # YOUR CODE HERE

# Test 1: single vector
x1 = np.array([1000., 1001., 1002.])
out1 = logsumexp(x1)
assert not np.isnan(out1) and not np.isinf(out1), 'should not overflow'
assert np.isclose(out1, 1002 + np.log(np.exp(-2) + np.exp(-1) + 1), atol=1e-6)

# Test 2: batch (N, C)
x2 = np.random.randn(8, 10)
out2 = logsumexp(x2)
assert out2.shape == (8,), out2.shape
# Verify against a direct NumPy reference on moderate inputs
ref2 = np.log(np.sum(np.exp(x2), axis=-1))
assert np.allclose(out2, ref2, atol=1e-6)

### D3 — log-softmax
Implement `log(softmax(x))` stably for `x` of shape `(N, C)` — use your `logsumexp`.

In [ ]:
def log_softmax(x):
    """x: (N, C). Returns (N, C)"""
    pass  # YOUR CODE HERE

x = np.random.randn(6, 5)
out = log_softmax(x)

assert out.shape == (6, 5)
# exp of log_softmax should be valid probabilities summing to 1
assert np.allclose(np.exp(out).sum(axis=1), 1.0, atol=1e-6)
# No NaN even on large inputs
big_x = np.array([[1000., 1001., 1002.]])
assert not np.any(np.isnan(log_softmax(big_x)))